# Addendum — LLM puro e retrieval por embedding

Adendo complementar a sequencia principal de experimentos (`results/`): duas variantes **novas**
comparadas offline contra os 180 resultados commitados, sem regenerá-los

- **pure_llm** — LLM puro, sem recuperação (contrafato do baseline RAG).
- **embedding** — retrieval semântico com `nomic-embed-text` (config de
  `configs/experiment.json` declara esse modelo desde o scaffold; era o
  gap lexical vs. semântico pendente da run oficial).

Mantendo hiperparametros igual a run principal: gerador `qwen2.5:7b`, `temperature: 0.0`,
`seed: 42`, `top_k: 4`, prompt de `prompts/generator.txt`, juiz
`llama3.1:8b`, 20 questões × 3 repetições, warmup por variante.

Artefatos novos: `results/pure_llm_r1..r3.csv`, `results/embedding_r1..r3.csv`,
`results/accuracy_addendum.csv`, `results/accuracy_addendum_summary.csv`,
`results/abstencoes_addendum.csv`, `results/traces/addendum_runs.jsonl`,
`results/addendum_manifest.json` e figuras `results/imagem/*_addendum.png`.

## Requisito antes de rodar

adições:
o fechamento **adiciona `answer_key` ao `questions.json` e `judge_model` ao
`configs/experiment.json`** — sem isso a célula 3 (config) interrompe o
notebook com mensagem explicativa.

## Como executar
1. Upload do PDF do livro em `corpus/documentos/`
   (`designing-machine-learning-systems-2022.pdf`, SHA-256 do
   `corpus/manifest.csv`).
2. Disponibilize o repositório completo em `/content/rag-dmls-lab`
   (clone da branch `feat/rag-experiment-evaluation`).
3. Execute as células em ordem — smoke test → run → grader → análise.
4. Devolva ao repositório os novos artefatos gerados em `results/` (CSVs e
   figuras `*_addendum` + `addendum_manifest.json`).


In [ ]:
# 0. Dependências do projeto (Colab)
# !pip install -q pypdf pandas numpy scikit-learn requests matplotlib openpyxl
import subprocess, sys
req = ['pypdf', 'pandas', 'numpy', 'scikit-learn', 'requests', 'matplotlib', 'openpyxl']
for pkg in req:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('dependências instaladas:', ', '.join(req))


In [ ]:
# 1. Ollama: instalação, servidor e modelos (mesmo padrão da run oficial)
import subprocess, time, os, requests

if not os.path.exists('/usr/local/bin/ollama'):
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

serve_log = open('/tmp/ollama_serve.log', 'w')
subprocess.Popen(['ollama', 'serve'], stdout=serve_log, stderr=serve_log)
time.sleep(3)

for model in ('qwen2.5:7b', 'llama3.1:8b', 'nomic-embed-text'):
    r = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
    assert r.returncode == 0, (model, r.stderr)
tags = requests.get('http://127.0.0.1:11434/api/tags', timeout=10).json()
present = sorted(m['name'] for m in tags['models'])
print('modelos Ollama:', present)


In [ ]:
# 2. Raiz do projeto, config, questions e PDF (validação de hashes)
from pathlib import Path
import json, hashlib, pandas as pd, numpy as np

def find_root():
    def has_markers(c):
        return (c / 'questions.json').exists() and (c / 'configs' / 'experiment.json').exists()
    c = Path.cwd()
    for cand in [c, *c.parents]:
        if has_markers(cand):
            return cand.resolve()
    raise FileNotFoundError('raiz do projeto não encontrada: rode em /content/rag-dmls-lab')

ROOT = find_root()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

config = json.loads((ROOT / 'configs' / 'experiment.json').read_text(encoding='utf-8'))
questions = json.loads((ROOT / 'questions.json').read_text(encoding='utf-8'))

# requisito 1: run já executada (status run_complete) e juiz declarado no config.
if 'judge_model' not in config or config.get('status') != 'run_complete':
    raise SystemExit(
        'configs/experiment.json sem judge_model/status run_complete — '
        'faz falta o commit/push da fase de fechamento nesta branch. '
        'Alternativa: subir localmente os configs/experiment.json, questions.json '
        'e results/ atualizados para /content/rag-dmls-lab e rodar este notebook.')

# answer_key precisa estar presente (commit da fase de fechamento)
missing_keys = [i['id'] for i in questions['questions'] if not str(i.get('answer_key', '')).strip()]
assert not missing_keys, f'answer_key ausente para: {missing_keys}'
assert len(questions['questions']) == config['question_count'] == 20
assert config['status'] == 'run_complete'

# corpus: mesmos arquivo e hash da run oficial
pdf_path = ROOT / 'corpus' / 'documentos'
pdfs = sorted(pdf_path.glob('*.pdf'))
assert len(pdfs) == 1, 'coloque exatamente o PDF do livro em corpus/documentos/'
pdf_path = pdfs[0]
corpus_sha = sha256_file(pdf_path)
manifest = pd.read_csv(ROOT / 'corpus' / 'manifest.csv')

# corpus: mesmo arquivo e hash da run oficial (referência: results/run_manifest.json;
# o hash de corpus/manifest.csv diverge da run — ver nota no README de results)
run_manifest = json.loads((ROOT / 'results' / 'run_manifest.json').read_text(encoding='utf-8'))
official_sha = run_manifest['corpus_sha256']
print('config:', config['project_name'], '| status:', config['status'])
print('PDF:', pdf_path.name, '| sha256:', corpus_sha[:16], '...')
assert corpus_sha == official_sha, (
    f'PDF difere da run oficial: esperado {official_sha[:16]}..., obtido {corpus_sha[:16]}...')
print('answer_key ok para 20 questões')


In [ ]:
# 3. Corpus: extração e chunking idênticos à run oficial (funções copiadas
# verbatim do notebook.ipynb oficial — mesmo 713 chunks esperado)
import re
from pypdf import PdfReader

def read_pdf_pages(pdf_path):
    """Retorna lista de (numero_real_da_pagina, texto) apenas para páginas com texto."""
    reader = PdfReader(pdf_path)
    pages = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = re.sub(r'\s+', ' ', page.extract_text() or '').strip()
        if text:
            pages.append((page_num, text))
    return pages

def chunk_pages_with_source(pages, chunk_size=800, overlap=120):
    """pages: lista de (numero_real, texto). Retorna (chunks, chunk_pages) com a página real."""
    full_text, page_starts = '', []
    for page_num, page_text in pages:
        page_starts.append((len(full_text), page_num))
        full_text += page_text + ' '

    chunks, chunk_pages = [], []
    step = chunk_size - overlap
    for start in range(0, len(full_text), step):
        piece = full_text[start:start + chunk_size].strip()
        if piece:
            page = max(num for pos, num in page_starts if pos <= start)
            chunks.append(piece)
            chunk_pages.append(page)
        if start + chunk_size >= len(full_text):
            break
    return chunks, chunk_pages

pages = read_pdf_pages(pdf_path)
chunks, chunk_pages = chunk_pages_with_source(pages, chunk_size=config['chunk_size_chars'], overlap=config['chunk_overlap_chars'])
print(f"Páginas com texto: {len(pages)}")
print(f"Chunks gerados: {len(chunks)} (run oficial: 713)")
assert len(chunks) == 713, 'chunking diverge da run oficial — confira o PDF (sha256)'


In [ ]:
# 4. Índice de embeddings (nomic-embed-text, via /api/embeddings em lote)
import requests, time

OLLAMA_URL = 'http://127.0.0.1:11434'
EMBED_MODEL = config['embedding_model']  # nomic-embed-text (proposto no config desde o scaffold)

def ollama_embed(texts, batch=32):
    vecs = []
    for i in range(0, len(texts), batch):
        batch_texts = texts[i:i + batch]
        payload = {'model': EMBED_MODEL, 'input': batch_texts}
        r = requests.post(f'{OLLAMA_URL}/api/embed', json=payload, timeout=120)
        r.raise_for_status()
        vecs.extend(r.json()['embeddings'])
    return np.array(vecs, dtype=np.float32)

t0 = time.perf_counter()
chunk_vecs = ollama_embed(chunks)
chunk_vecs /= np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
print(f'matrix de embeddings: {chunk_vecs.shape} em {time.perf_counter() - t0:.1f}s')

def embed_query(query):
    q = ollama_embed([query])[0]
    return q / np.linalg.norm(q)

def retrieve_top_k_embedding(query, chunks, chunk_vecs, k=4):
    """Retrieval semântico: cosseno query-chunk, top-k com score > 0."""
    q = embed_query(query)
    scores = chunk_vecs @ q
    ranked = [idx for idx in np.argsort(scores)[::-1] if scores[idx] > 0][:k]
    return [(chunks[idx], float(scores[idx])) for idx in ranked]


In [ ]:
# 5. Geração: call_ollama (igual à oficial) e format_prompt (versionado em prompts/generator.txt)
import requests as _rq

GEN_PATH = ROOT / 'prompts' / 'generator.txt'

def load_generator_template(path=None):
    """Template da resposta versionado em prompts/generator.txt (fallback inline = texto da run)."""
    path = Path(path) if path else GEN_PATH
    if path.exists():
        return path.read_text(encoding='utf-8')
    return ('Answer the question using only the provided context. '
            'Answer in English. If the context does not contain enough evidence, say so explicitly.\n\n'
            'Question: {question}\n\nContext:\n{context}\n\nAnswer:')

def format_prompt(question, context):
    return load_generator_template().format(question=question, context=context)

def format_prompt_no_context(question):
    """LLM puro: a mesma instrução do gerador, sem bloco de contexto.
    O modelo deve responder com o próprio conhecimento e dizer quando não sabe."""
    return ('Answer the question using your own knowledge. '
            'Answer in English. If you do not know the answer, say so explicitly.\n\n'
            f'Question: {question}\n\nAnswer:')

def build_context(passages):
    return '\n\n---\n\n'.join(
        f'[Passage {i + 1}] {text[:1600]}' for i, (text, _) in enumerate(passages)
    )

def call_ollama(prompt, model=None, temperature=None, timeout=120):
    payload = {
        'model': model or config['generator_model'],
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': temperature if temperature is not None
                           else config['generation_temperature'],
            'seed': config['seed'],
        },
    }
    response = _rq.post(f'{OLLAMA_URL}/api/generate', json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json().get('response', '').strip()

def answer_question_pure_llm(question, model=None, hits=None):
    """LLM puro, sem RAG: apenas a pergunta, sem evidência recuperada (hits ignorado)."""
    try:
        return call_ollama(format_prompt_no_context(question), model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'

def answer_question_embedding(question, model=None, hits=None):
    """Retrieval semântico (nomic-embed-text) + a mesma geração do baseline."""
    try:
        hits = hits if hits is not None else retrieve_top_k_embedding(
            question, chunks, chunk_vecs, k=config['top_k'])
        if not hits:
            return 'Não encontrei evidência suficiente no corpus para responder com confiança.'
        return call_ollama(format_prompt(question, build_context(hits[:config['top_k']])), model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'


In [ ]:
# 6. Smoke test: 5 primeiras questões × 2 variantes novas, contratos do smoke oficial
ABSTENTION_RE = re.compile(
    r"(not enough evidence|does not contain|insufficient|no initial evidence|"
    r"cannot provide|cannot answer|say so explicitly|do not know|don't know|não encontrei)", re.I)

def run_smoke(n=5):
    rows = []
    for item in questions['questions'][:n]:
        question = item['question']
        for variant, fn in (('pure_llm', answer_question_pure_llm),
                            ('embedding', answer_question_embedding)):
            start = time.perf_counter()
            if variant == 'embedding':
                hits = retrieve_top_k_embedding(question, chunks, chunk_vecs, k=config['top_k'])
            else:
                hits = []
            answer = fn(question, hits=hits)
            rows.append({'id': item['id'], 'variant': variant,
                         'latency_sec': round(time.perf_counter() - start, 3), 'answer': answer})
    return pd.DataFrame(rows)

sample_df = run_smoke()
assert len(sample_df) == 10
assert sample_df['answer'].str.strip().ne('').all(), 'resposta vazia'
assert not sample_df['answer'].str.startswith('Erro ao chamar o Ollama').any(), 'Ollama não respondeu'
assert sample_df['latency_sec'].between(0, 600).all(), 'latência fora do intervalo'
print('SMOKE TEST PASSOU')
print(sample_df.groupby('variant')['latency_sec'].describe()[['mean', 'min', 'max']])


In [ ]:
# 7. Run addendum: pure_llm e embedding, 20 questões × 3 repetições, traces e manifest
from datetime import datetime, timezone

RESULTS_DIR = ROOT / 'results'
TRACES_DIR = RESULTS_DIR / 'traces'
TRACES_DIR.mkdir(exist_ok=True)

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

CONFIG_ADD_SHA = sha256_bytes(json.dumps(config, sort_keys=True).encode())
QUESTIONS_ADD_SHA = sha256_bytes((ROOT / 'questions.json').read_bytes())

_add_orig_call = call_ollama
_calls = {'n': 0}
def call_ollama(prompt, *args, **kwargs):  # conta chamadas sem duplicar do notebook oficial
    _calls['n'] += 1
    return _add_orig_call(prompt, *args, **kwargs)

def warmup():
    call_ollama('Reply with the single word: OK.')

def hits_for(variant, question):
    if variant == 'embedding':
        return retrieve_top_k_embedding(question, chunks, chunk_vecs, k=config['top_k'])
    return []

def run_addendum_experiment():
    with open(TRACES_DIR / 'addendum_runs.jsonl', 'w', encoding='utf-8') as trace:
        for variant, reps in (('pure_llm', 3), ('embedding', 3)):
            warmup()
            rows = []
            for repetition in range(1, reps + 1):
                for item in questions['questions']:
                    question = item['question']
                    _calls['n'] = 0
                    start = time.perf_counter()
                    hits = hits_for(variant, question)
                    answer = answer_question_pure_llm(question) if variant == 'pure_llm' \
                        else answer_question_embedding(question, hits=hits)
                    latency = round(time.perf_counter() - start, 3)
                    row = {'question_id': item['id'], 'category': item['category'],
                           'difficulty': item['difficulty'], 'variant': variant,
                           'repetition': repetition, 'route_question': variant,
                           'latency_sec': latency, 'evidence_count': len(hits),
                           'llm_calls': _calls['n'], 'answer': answer}
                    rows.append(row)
                    ev = ([{'rank': r + 1, 'text': t, 'score': round(s, 6)}
                           for r, (t, s) in enumerate(hits)] if variant == 'embedding' else [])
                    trace.write(json.dumps({**row, 'evidence': ev}, ensure_ascii=False) + '\n')
            df = pd.DataFrame(rows)
            for r in range(1, 4):
                df[df['repetition'] == r].to_csv(
                    RESULTS_DIR / f'{variant}_r{r}.csv', index=False)
            print(f'{variant}: {len(df)} respostas gravadas')
    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'kind': 'addendum',
        'generator_model': config['generator_model'],
        'judge_model': config['judge_model'],
        'embedding_model': config['embedding_model'],
        'top_k': config['top_k'],
        'temperature': config['generation_temperature'],
        'seed': config['seed'],
        'chunk_count': len(chunks),
        'question_count': len(questions['questions']),
        'variants': {'pure_llm': 3, 'embedding': 3},
        'warmup_per_variant': True,
        'abstention_regex': ABSTENTION_RE.pattern,
        'config_sha256': CONFIG_ADD_SHA,
        'questions_sha256': QUESTIONS_ADD_SHA,
        'corpus_sha256': corpus_sha,
        'trace_file': 'results/traces/addendum_runs.jsonl',
        'note': 'Adendo à run oficial de 2026-10-15; não sobrescreve CSVs oficiais.',
    }
    (RESULTS_DIR / 'addendum_manifest.json').write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('run addendum concluída')

run_addendum_experiment()  # executa a run completa do adendo


In [ ]:
# 8. Grader addendum: mesmo contrato do juiz da run oficial (llama3.1:8b, 1 palavra)
JUDGE_MODEL = config['judge_model']

def parse_verdict(raw):
    m = re.search(r'\b(INCORRECT|NOT CORRECT|CORRECT)\b', raw.upper())
    if not m:
        return 'PARSE_ERROR'
    return 'CORRECT' if m.group(1) == 'CORRECT' else 'INCORRECT'

def grade_answer(question, gold, answer):
    prompt = (
        "You are a strict grader. Decide if the ANSWER conveys the same core "
        "information as the GOLD ANSWER for the question below.\n"
        "Reply with exactly one word: CORRECT or INCORRECT. No explanation.\n\n"
        f"Question: {question}\n\n"
        f"GOLD ANSWER: {gold}\n\n"
        f"ANSWER: {answer}\n\n"
        "VERDICT:"
    )
    return parse_verdict(call_ollama(prompt, model=JUDGE_MODEL))

key_map = {i['id']: str(i['answer_key']).strip() for i in questions['questions']}
q_map = {i['id']: i['question'] for i in questions['questions']}

addendum_df = pd.concat(
    [pd.read_csv(RESULTS_DIR / f'pure_llm_r{r}.csv') for r in (1, 2, 3)]
    + [pd.read_csv(RESULTS_DIR / f'embedding_r{r}.csv') for r in (1, 2, 3)],
    ignore_index=True)
for v in ('pure_llm', 'embedding'):
    assert len(addendum_df[addendum_df['variant'] == v]) == 60, f'run incompleta: {v}'

warmup()
grade_rows = []
for idx, (_, row) in enumerate(addendum_df.iterrows(), start=1):
    grade_rows.append({
        'question_id': row['question_id'], 'variant': row['variant'],
        'repetition': row['repetition'], 'difficulty': row['difficulty'],
        'verdict': grade_answer(q_map[row['question_id']], key_map[row['question_id']], str(row['answer'])),
    })
    if idx % 30 == 0:
        print(f'{idx}/{len(addendum_df)} respostas avaliadas')

grades_add = pd.DataFrame(grade_rows)
grades_add.to_csv(RESULTS_DIR / 'accuracy_addendum.csv', index=False)

summary_add = grades_add.groupby('variant', as_index=False).agg(
    accuracy=('verdict', lambda v: (v == 'CORRECT').mean().round(3)),
    parse_errors=('verdict', lambda v: int((v == 'PARSE_ERROR').sum())),
    n=('verdict', 'count'))
summary_add.to_csv(RESULTS_DIR / 'accuracy_addendum_summary.csv', index=False)
print(summary_add.to_string(index=False))


In [ ]:
# 9. Abstenções do addendum (mesma regex reconstruída da run oficial)
addendum_df['abstencao'] = addendum_df['answer'].astype(str).apply(lambda x: bool(ABSTENTION_RE.search(x)))
grades_add = grades_add.merge(addendum_df[['question_id', 'variant', 'repetition', 'abstencao']],
                              on=['question_id', 'variant', 'repetition'])

rows = []
for v in ('pure_llm', 'embedding'):
    sub = addendum_df[addendum_df['variant'] == v]
    gsub = grades_add[grades_add['variant'] == v]
    ab = sub['abstencao']
    resp_ok = gsub.loc[~ab.values, 'verdict'].eq('CORRECT')
    rows.append({'variant': v, 'n_execucoes': len(sub), 'n_abstencoes': int(ab.sum()),
                 'taxa_abstencao': round(ab.mean(), 4), 'n_respondido': int((~ab).sum()),
                 'corretas_quando_responde': int(resp_ok.sum()),
                 'precisao_se_responde': round(resp_ok.mean(), 4) if len(resp_ok) else None})
ab_add = pd.DataFrame(rows)
ab_add.to_csv(RESULTS_DIR / 'abstencoes_addendum.csv', index=False)
print(ab_add.to_string(index=False))


In [ ]:
# 10. Comparação final: 5 variantes (3 oficiais + 2 addendum) — tabela consolidada
import matplotlib.pyplot as plt
import seaborn as sns

IMAGE_DIR = RESULTS_DIR / 'imagem'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

official = pd.concat(
    [pd.read_csv(RESULTS_DIR / f'baseline_r{r}.csv') for r in (1, 2, 3)]
    + [pd.read_csv(RESULTS_DIR / 'results_adaptive.csv')]
    + [pd.read_csv(RESULTS_DIR / f'self_rag_r{r}.csv')], ignore_index=True)
official = official.merge(pd.read_csv(RESULTS_DIR / 'accuracy.csv')[['question_id', 'variant', 'repetition', 'verdict']],
                          on=['question_id', 'variant', 'repetition'], how='left')
addendum_df = addendum_df.merge(
    grades_add[['question_id', 'variant', 'repetition', 'verdict']],
    on=['question_id', 'variant', 'repetition'], how='left')
assert official['verdict'].notna().all() and addendum_df['verdict'].notna().all()

all_df = pd.concat([official, addendum_df], ignore_index=True)
ORDER = ['baseline', 'adaptive', 'self_rag', 'embedding', 'pure_llm']

def variant_stats(df):
    out = []
    for v in ORDER:
        sub = df[df['variant'] == v]
        ab = sub['answer'].astype(str).apply(lambda x: bool(ABSTENTION_RE.search(x)))
        resp_correct = int(sub.loc[~ab, 'verdict'].eq('CORRECT').sum())
        out.append({
            'variant': v, 'n': len(sub),
            'accuracy': round((sub['verdict'] == 'CORRECT').mean(), 3),
            'taxa_abstencao': round(ab.mean(), 3),
            'corretas_quando_responde': f"{resp_correct}/{int((~ab).sum())}",
            'latencia_media_s': round(sub['latency_sec'].mean(), 3),
            'llm_calls_medio': round(sub['llm_calls'].mean(), 2),
            'evidence_medio': round(sub['evidence_count'].mean(), 2),
        })
    return pd.DataFrame(out)

final_cmp = variant_stats(all_df)
final_cmp.to_csv(RESULTS_DIR / 'comparacao_variaveis_addendum.csv', index=False)
print(final_cmp.to_string(index=False))
PALETTE = {'baseline': '#8da0cb', 'adaptive': '#66c2a5', 'self_rag': '#fc8d62',
           'embedding': '#e78ac3', 'pure_llm': '#a6d854'}


In [ ]:
# 11. Figuras do addendum: acurácia, latência, chamadas e abstenção nas 5 variantes
fig, axes = plt.subplots(1, 4, figsize=(22, 5.2))

sns.barplot(data=final_cmp, x='variant', y='accuracy', order=ORDER, hue='variant',
            hue_order=ORDER, palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title('Acurácia por variante (juiz não validado)')

sns.barplot(data=final_cmp, x='variant', y='latencia_media_s', order=ORDER, hue='variant',
            hue_order=ORDER, palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_title('Latência média por variante')

sns.barplot(data=final_cmp, x='variant', y='llm_calls_medio', order=ORDER, hue='variant',
            hue_order=ORDER, palette=PALETTE, legend=False, ax=axes[2])
axes[2].set_title('Chamadas LLM médias por variante')

sns.barplot(data=final_cmp, x='variant', y='taxa_abstencao', order=ORDER, hue='variant',
            hue_order=ORDER, palette=PALETTE, legend=False, ax=axes[3])
axes[3].set_ylim(0, 1)
axes[3].set_title('Taxa de abstenção (heurística lexical)')

for ax in axes:
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
out_png = IMAGE_DIR / 'comparacao_addendum.png'
fig.savefig(out_png, dpi=150)
plt.show()
assert out_png.exists()
print('figura gravada em:', out_png)


In [ ]:
# boxplot de latência com as 5 variantes
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=all_df, x='variant', y='latency_sec', order=ORDER, hue='variant',
            hue_order=ORDER, palette=PALETTE, legend=False, ax=ax)
ax.set_title('Distribuição de latência por variante — run oficial + adendo')
fig.tight_layout()
out_png = IMAGE_DIR / 'boxplot_latencia_addendum.png'
fig.savefig(out_png, dpi=150)
plt.show()
assert out_png.exists()
print('figura gravada em:', out_png)


#### Conclusões esperadas desta run (preencher após execução)

**Como ler.** O adendo cria duas variantes de controle: `pure_llm` (sem
recuperação) mostra o que o gerador já sabe sem o corpus — piso que valida
a answer key; `embedding` isola a contribuição da recuperação semântica
(diferença única: TF-IDF → cosseno de embeddings) sobre o baseline RAG.

**Preencher após a run**, mantendo o padrão da célula de conclusões oficial:
comparar `accuracy` de cada variante com as 3 oficiais (tabela
`comparacao_variaveis_addendum.csv`); separar sinal de ruído olhando as 3
repetições; declarar se o ganho vem de abstenção ou de recuperação melhor,
usando `abstencoes_addendum.csv` e `comparacao_variaveis_addendum.csv`;
citar limitações (juiz não validado; composição de embeddings não
normalizada para chunks longos; chunks idênticos à run oficial).

**Não regenerar** os CSVs oficiais: os artefatos do adendo são novos
arquivos, nunca sobrescritos.
